<div dir="rtl">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ariefouren/ONO-linear-algebra-2026/blob/main/03_Chapters/03_Matrices/notebooks/03.04_invertible_and_elementary.ipynb)

# אלגברה לינארית – פרק 3: מטריצות
## מחברת 03.04: מטריצות הפיכות ומטריצות אלמנטריות

מחברת זו מלווה את **סיכום 03.04** (סעיפים 3.8 ו-3.9 בספר). הספר הרשמי של הקורס: פארן וליבוביץ, *אלגברה לינארית 1, כרך א*, האוניברסיטה הפתוחה, 2019 (זמין בספרייה הדיגיטלית של הקריה האקדמית אונו, מאגר למדא).

**איך משתמשים:** בתפריט *Runtime* בוחרים *Run all*, או מריצים תא-תא עם Shift+Enter. מומלץ לשנות את המספרים בתאים ולהריץ שוב – כך רואים מה כל פעולה עושה.

הכנה: ד"ר אריה פורן, הקריה האקדמית אונו, תשפ"ז.

</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(suppress=True, precision=4)

def show(name, M):
    """Print a matrix with its name and its order (rows x columns)."""
    M = np.asarray(M)
    print(f"{name}  ({M.shape[0]}x{M.shape[1] if M.ndim > 1 else 1})")
    print(M, end="\n\n")

<div dir="rtl">

## 1. הופכית ופתרון מערכת (דוגמה 3.16)

</div>

In [ ]:
A = np.array([[2, 1], [5, 3]])
B = np.array([[3, -1], [-5, 2]])
show("AB", A @ B); show("BA", B @ A)
show("np.linalg.inv(A)", np.linalg.inv(A))
b = np.array([4, 7])
print("x = A^{-1} b =", B @ b, "  check:", A @ (B @ b))
print("np.linalg.solve(A, b) =", np.linalg.solve(A, b), " (the recommended way)")

<div dir="rtl">

## 2. מטריצה סינגולרית

למטריצה עם שורת אפסים (או שורה שהיא צירוף של אחרות) אין הופכית.

</div>

In [ ]:
S = np.array([[1, 0, 2], [2, 1, 3], [0, 1, -1]])
try:
    np.linalg.inv(S)
except np.linalg.LinAlgError as e:
    print("LinAlgError:", e)
print("row 3 == row 2 - 2*row 1 ?", np.array_equal(S[2], S[1] - 2*S[0]))

<div dir="rtl">

## 3. תכונות ההופכית (משפט 3.8.4)

$(AB)^{-1}=B^{-1}A^{-1}$ ו-$(A^t)^{-1}=(A^{-1})^t$.

</div>

In [ ]:
inv = np.linalg.inv
A = np.array([[1, 2], [1, 3]]); B = np.array([[2, 0], [1, 1]])
show("(AB)^-1", inv(A @ B)); show("B^-1 A^-1", inv(B) @ inv(A)); show("A^-1 B^-1 (wrong order)", inv(A) @ inv(B))
print("(A^t)^-1 == (A^-1)^t :", np.allclose(inv(A.T), inv(A).T))

<div dir="rtl">

## 4. מטריצות אלמנטריות (הגדרה 3.9.1, טענה 3.9.3)

נבנה את שלוש הפעולות האלמנטריות כפונקציות שמחזירות את $\varphi(I)$, ונבדוק ש-$\varphi(A)=\varphi(I)A$.

</div>

In [ ]:
def E_swap(n, i, j):            # R_i <-> R_j   (i, j start at 1)
    E = np.eye(n); E[[i-1, j-1]] = E[[j-1, i-1]]; return E
def E_scale(n, i, c):           # R_i -> c R_i, c != 0
    E = np.eye(n); E[i-1, i-1] = c; return E
def E_add(n, i, j, c):          # R_i -> R_i + c R_j
    E = np.eye(n); E[i-1, j-1] = c; return E

A = np.array([[1, 4], [0, 3], [2, 5]], dtype=float)
E = E_add(3, 3, 1, -2)          # R3 -> R3 - 2 R1
show("phi(I)", E)
show("phi(I) A", E @ A)
A2 = A.copy(); A2[2] = A2[2] - 2*A2[0]
print("same as doing the row operation directly:", np.array_equal(E @ A, A2))
show("inverse of phi(I) = phi^-1(I):  R3 -> R3 + 2 R1", np.linalg.inv(E))

<div dir="rtl">

## 5. האלגוריתם $[A\,|\,I]\to[I\,|\,A^{-1}]$ צעד אחר צעד (דוגמה 3.18)

הפונקציה מדפיסה כל פעולת שורה. משתמשים ב-`sympy` כדי לעבוד בשברים מדויקים.

</div>

In [ ]:
import sympy as sp

def inverse_steps(A):
    A = sp.Matrix(A); n = A.rows
    M = A.row_join(sp.eye(n))
    print("start [A | I]:"); sp.pprint(M); print()
    for c in range(n):
        p = next((r for r in range(c, n) if M[r, c] != 0), None)
        if p is None:
            print(f"no pivot in column {c+1}: A is NOT invertible"); return None
        if p != c:
            M.row_swap(p, c); print(f"R{c+1} <-> R{p+1}"); sp.pprint(M); print()
        if M[c, c] != 1:
            k = M[c, c]; M[c, :] = M[c, :] / k; print(f"R{c+1} -> (1/({k})) R{c+1}"); sp.pprint(M); print()
        for r in range(n):
            if r != c and M[r, c] != 0:
                k = M[r, c]; M[r, :] = M[r, :] - k*M[c, :]
                coef = "" if abs(k) == 1 else f"{abs(k)} "
                sign = "-" if k > 0 else "+"
                print(f"R{r+1} -> R{r+1} {sign} {coef}R{c+1}"); sp.pprint(M); print()
    inv = M[:, n:]
    print("A^-1 ="); sp.pprint(inv); print("check A*A^-1 == I:", A*inv == sp.eye(n))
    return inv

inverse_steps([[1, 2, 0], [0, 1, 1], [1, 0, -1]]);

<div dir="rtl">

ההופכית יצאה זהה לזו שבסיכום, אף שסדר הפעולות כאן קצת שונה – **סדר הפעולות אינו משנה את התוצאה הסופית**. ועכשיו מטריצה שאינה הפיכה (דוגמה 3.19):

</div>

In [ ]:
inverse_steps([[1, 0, 2], [2, 1, 3], [0, 1, -1]]);

<div dir="rtl">

## 6. הצגה כמכפלה של מטריצות אלמנטריות (דוגמה 3.20)

</div>

In [ ]:
E1inv = E_add(2, 2, 1, 3)       # R2 -> R2 + 3 R1
E2inv = E_scale(2, 2, -2)       # R2 -> -2 R2
E3inv = E_add(2, 1, 2, 2)       # R1 -> R1 + 2 R2
show("E1^-1 E2^-1 E3^-1", E1inv @ E2inv @ E3inv)

<div dir="rtl">

## 7. יישום: לבטל טרנספורמציה בגרפיקה

$(SR)^{-1}=R^{-1}S^{-1}$: קודם מבטלים את המתיחה, אחר כך את הסיבוב.

</div>

In [ ]:
P = np.array([[0, 2, 2, 1, 0, 0], [0, 0, 2, 3, 2, 0]], dtype=float)
R = np.array([[0, -1], [1, 0]]); S = np.array([[2, 0], [0, 1]])
Q = S @ R @ P
back = np.linalg.inv(R) @ np.linalg.inv(S) @ Q
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for a, M, t in zip(ax, [P, Q, back], ["P", "SRP", "R^-1 S^-1 (SRP)"]):
    a.plot(M[0], M[1], "-o"); a.set_title(t); a.set_aspect("equal"); a.grid(True); a.set_xlim(-7, 5); a.set_ylim(-3, 5)
plt.show()
print("restored exactly:", np.allclose(back, P))

<div dir="rtl">

## 8. יישום: הצפנה פשוטה בעזרת מטריצה הפיכה

כל זוג אותיות הופך לווקטור מספרים ($A=1,\dots,Z=26$), וכופלים במטריצת מפתח $K$. המפענח כופל ב-$K^{-1}$. (בצופן היל האמיתי החשבון נעשה מודולו 26.)

</div>

In [ ]:
K = np.array([[3, 1], [5, 2]])            # key (invertible)
Kinv = np.linalg.inv(K)

def encode(text):
    nums = [ord(ch) - 64 for ch in text.upper() if ch.isalpha()]
    if len(nums) % 2: nums.append(24)     # pad with 'X'
    return K @ np.array(nums).reshape(-1, 2).T

def decode(C):
    nums = np.rint(Kinv @ C).astype(int).T.ravel()
    return "".join(chr(n + 64) for n in nums)

C = encode("LINEAR ALGEBRA")
print("cipher:\n", C)
print("decoded:", decode(C))

<div dir="rtl">

## 9. יישום: פירוק LU

תוכנות מדעיות שומרות את צעדי הדירוג כפירוק $A=PLU$ ($L$ משולשית תחתונה, $U$ משולשית עליונה).

</div>

In [ ]:
from scipy.linalg import lu
A = np.array([[1, 2, 0], [0, 1, 1], [1, 0, -1]], dtype=float)
Pm, L, U = lu(A)
show("L", L); show("U", U)
print("P L U == A:", np.allclose(Pm @ L @ U, A))

<div dir="rtl">

## 10. בדקו את עצמכם

</div>

In [ ]:
inverse_steps([[3, 1], [5, 2]]);
print("x =", list(sp.Matrix([[2, -1], [-5, 3]]) * sp.Matrix([1, 0])))
inverse_steps([[1, 2, 3], [2, 4, 7], [1, 2, 5]]);